# ODSC 2026 — Notebook 2: OpenClaw

Custom Claw showed you what an agent looks like under the hood. OpenClaw is what you ship.

| Part | What you build |
| --- | --- |
| **Part 1** | Start OpenClaw, review `AGENTS.md`, run your first compliance check |
| **Part 2** | Telegram notifications — pair your bot and get alerts on your phone |
| **Part 3** | Observability with Opik — traces, token costs, tool timelines |

**Remember- Functions are code, skills are natural language

**Prerequisites:** Completed `1_custom_claw.ipynb`. Docker Desktop is running.

In [ ]:
import subprocess
import os
import shutil
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("your-"), "Set OPENAI_API_KEY in .env"

docker_ok = subprocess.run("docker info".split(), capture_output=True).returncode == 0
assert docker_ok, "Docker is not running. Start Docker Desktop and try again."

# Auto-detect Docker sudo/compose variant
SUDO = ""
if subprocess.run("docker info".split(), capture_output=True).returncode != 0:
    if subprocess.run("sudo docker info".split(), capture_output=True).returncode == 0:
        SUDO = "sudo "

DOCKER_COMPOSE = (
    f"{SUDO}docker compose"
    if subprocess.run(f"{SUDO}docker compose version".split(), capture_output=True).returncode == 0
    else f"{SUDO}docker-compose"
)
OPENCLAW_COMPOSE = f"{DOCKER_COMPOSE} -f openclaw-agent/docker-compose.yml"

shutil.copy(".env", "openclaw-agent/.env")

print("✅ OpenAI key set")
print("✅ Docker running")
print(f"Using: {DOCKER_COMPOSE}")
print("\nReady to go.")

---
## Part 1: Start OpenClaw

---
### Why OpenClaw?

Custom Claw is a pydantic-ai agent you run manually in a notebook. It fetches pages, detects changes, and reports findings — but only when you run it. There's no scheduling, no persistent conversation, and no way to query it without opening the notebook.

OpenClaw is an always-on agent framework that runs in Docker. It wraps the same compliance-checking logic with:
- Persistent conversation memory across sessions
- Web dashboard for monitoring
- Telegram notifications with a simple pairing flow
- Skills defined as Markdown + shell scripts — no Python required

The `check-compliance-updates` skill in `openclaw-agent/skills/` is the compliance checking logic. Your `AGENTS.md` is what gives OpenClaw its domain expertise.

### Building `AGENTS.md` — Your Agent's Domain Knowledge

With Custom Claw, domain knowledge lives in the code: a hardcoded system prompt, hardcoded URLs, hardcoded logic.

With OpenClaw, you externalize that knowledge into `AGENTS.md` — a plain text file the agent reads as context on every turn. It defines:
- **What it does** — its purpose and scope
- **Domain knowledge** — teacher licensure rules, Praxis exams, key sources by state
- **Skills available** — which tools it can call and when to use each
- **Behavior guidelines** — how to respond, what to cite, when to search

Open `AGENTS.md` in the project root and read through it before running OpenClaw — that file is where the agent's intelligence comes from.

> **The pattern:** Define the domain, the tools, and the behavioral guardrails *before* you write a line of code. The agent reasons from this context.

In [ ]:
import shutil
import time

OPENCLAW_COMPOSE = f"{DOCKER_COMPOSE} -f openclaw-agent/docker-compose.yml"

# Sync .env
shutil.copy(".env", "openclaw-agent/.env")

print("Starting OpenClaw gateway...")
!{OPENCLAW_COMPOSE} up -d openclaw-gateway

for i in range(15):
    health = subprocess.run(
        f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
        capture_output=True
    )
    if health.returncode == 0:
        print("✅ Gateway healthy!")
        break
    time.sleep(3)
    print(f"   Waiting... ({i+1})")

In [ ]:
# Configure the agent
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set agents.defaults.model openai/gpt-4o
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set gateway.bind lan
print("\n✅ Agent configured.")

In [ ]:
import json as _json

cmd = OPENCLAW_COMPOSE.split() + [
    "exec", "openclaw-gateway", "node", "/app/openclaw.mjs",
    "agent", "--agent", "main", "--json",
    "-m", "Check for any teacher licensure compliance updates in Ohio and Texas.",
]
result = subprocess.run(cmd, capture_output=True, text=True)

try:
    data = _json.loads(result.stdout)
    print(data["result"]["payloads"][0]["text"])
except Exception:
    print(result.stdout or result.stderr)

In [ ]:
# Open the OpenClaw web dashboard
result = subprocess.run(
    OPENCLAW_COMPOSE.split() + [
        "exec", "openclaw-gateway", "node", "/app/openclaw.mjs", "dashboard", "--no-open"
    ],
    capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())
print("""
1. Click the link above to open the dashboard in your browser
2. Come back here and run the next cell to approve the connection""")

In [ ]:
import re

# Approve the browser's device pairing request
result = subprocess.run(
    OPENCLAW_COMPOSE.split() + [
        "exec", "openclaw-gateway", "node", "/app/openclaw.mjs", "devices", "list"
    ],
    capture_output=True, text=True
)

uuids = re.findall(r'Pending.*?([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})',
                   result.stdout, re.DOTALL)

if not uuids:
    print("No pending device requests — dashboard may already be connected.")
else:
    request_id = uuids[0]
    approve = subprocess.run(
        OPENCLAW_COMPOSE.split() + [
            "exec", "openclaw-gateway", "node", "/app/openclaw.mjs",
            "devices", "approve", request_id
        ],
        capture_output=True, text=True
    )
    print(approve.stdout.strip() or approve.stderr.strip())
    print("\n✅ Dashboard approved — reload the browser tab.")

---
## Part 2: Telegram Notifications

### Setup

1. Open Telegram and search for **@BotFather** → send `/newbot` → follow the prompts → copy the token
2. Add it to your `.env`:
   ```
   OPENCLAW_TELEGRAM_BOT_TOKEN=your-token-here
   ```
3. Run the first cell below — it registers your bot with OpenClaw
4. Open Telegram and send your bot any message — it will reply with a **pairing code**
5. Run the second cell, enter the pairing code when prompted, and hit Enter
6. Send your bot another message — you're connected

Done! Congratulations on your new bot. You will find it at t.me/Odscwsbot. You can now add a description, about section and profile picture for your bot, see /help for a list of commands. By the way, when you've finished creating your cool bot, ping our Bot Support if you want a better username for it. Just make sure the bot is fully operational before you do this.

Use this token to access the HTTP API:
8696399487:AAGC8gF7N0s1u9_IHWxprHFMowpG5MAzIIg
Keep your token secure and store it safely, it can be used by anyone to control your bot.

For a description of the Bot API, see this page: https://core.telegram.org/bots/api

In [ ]:
from dotenv import find_dotenv, dotenv_values

_env = dotenv_values(find_dotenv())
TELEGRAM_BOT_TOKEN = _env.get("OPENCLAW_TELEGRAM_BOT_TOKEN", "")
assert TELEGRAM_BOT_TOKEN, "Set OPENCLAW_TELEGRAM_BOT_TOKEN in .env first"

print("Adding Telegram channel to OpenClaw...")
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs channels add --channel telegram --token {TELEGRAM_BOT_TOKEN}

print("\n✅ Telegram channel added.")
print("Open Telegram, find your bot, and send it any message — OpenClaw is listening.")

In [ ]:
# After messaging your bot, enter the pairing code it sends back
code = input("Pairing code from Telegram: ").strip()
assert code, "No pairing code entered"

result = subprocess.run(
    OPENCLAW_COMPOSE.split() + [
        "exec", "openclaw-gateway", "node", "/app/openclaw.mjs",
        "pairing", "approve", "telegram", code,
    ],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

Hello

How can I assist you today?

---
### Try It — Example Prompts

Send any of these to your bot in Telegram:

**Compliance checks**
- `Check for teacher licensure updates in Ohio`
- `Have Praxis requirements changed in Texas recently?`
- `Check all monitored states for compliance changes`
- `What are the certification requirements for Math 7-12 in California?`

**Search**
- `Search for recent news about teacher certification changes in Florida`
- `Find the Indiana Department of Education licensure page`

**Scheduled tasks**
- `Send me a fun fact about education every day at 9am`
- `Every Monday morning, check all monitored states and send me a compliance summary`
- `Remind me every Friday at 4pm to review any pending certification renewals`
- `Search for education policy news every Sunday and send me a digest`

**Just for fun**
- `What's the weather in my city?`
- `Tell me something interesting about the history of teacher certification in the US`
- `Search for the most unusual teacher certification requirement in any US state`

**Create a new skill on the fly**
- `Create a skill that checks the ETS Praxis website for score requirement changes`
- `Write a skill that monitors NASDTEC for interstate agreement updates`

> **Skills vs. `AGENTS.md`:** Skills are tool definitions — scripts in `openclaw-agent/skills/` the agent can call. `AGENTS.md` is domain knowledge — it tells the agent *what it knows* and *how to behave*. They work together: `AGENTS.md` lists which skills exist; each skill's `SKILL.md` tells the agent when and how to use it.

# Back to the Slides

---
## Part 3: Observability with Opik

OpenClaw is running, but we have no visibility into *how* it's making decisions — which tools it called, how long they took, or what it cost.

[`@opik/opik-openclaw`](https://github.com/comet-ml/opik-openclaw) is the official OpenClaw plugin for Opik. It exports full agent traces: LLM spans, tool call spans with inputs/outputs, sub-agent spans, and run-level cost metadata.

The plugin is pre-installed in this repo. You just need to wire up your credentials.

### Setup

1. Sign up at [comet.com/opik](https://www.comet.com/opik) (free tier available)
2. Add these two keys to your `.env`:
   ```
   OPIK_API_KEY=your-api-key
   OPIK_WORKSPACE=your-username
   ```
3. Run the install cell, then the configure cell — they wire the plugin into OpenClaw and restart the gateway
4. Run the status cell to confirm everything is connected
5. Send your bot a message and watch the trace appear in your Opik dashboard under the `compliance-agent` project

In [ ]:
# Install the Opik plugin via npm (bypasses ClaHub rate limits)
print("Installing @opik/opik-openclaw...")

npm = subprocess.run(
    OPENCLAW_COMPOSE.split() + [
        "exec", "openclaw-gateway", "sh", "-c",
        "npm install --prefix /home/node/.openclaw/extensions @opik/opik-openclaw 2>&1 | tail -1"
    ],
    capture_output=True, text=True
)

register = subprocess.run(
    OPENCLAW_COMPOSE.split() + [
        "exec", "openclaw-gateway", "node", "/app/openclaw.mjs",
        "plugins", "install",
        "/home/node/.openclaw/extensions/node_modules/@opik/opik-openclaw"
    ],
    capture_output=True, text=True
)

if "Installed plugin" in register.stdout:
    print("✅ Opik plugin installed.")
elif "already" in register.stdout.lower():
    print("✅ Opik plugin already installed.")
else:
    print(register.stdout or register.stderr)

In [ ]:
from dotenv import find_dotenv, dotenv_values

_env = dotenv_values(find_dotenv())
api_key   = _env.get("OPIK_API_KEY", "")
workspace = _env.get("OPIK_WORKSPACE", "")
assert api_key,   "Set OPIK_API_KEY in .env first"
assert workspace, "Set OPIK_WORKSPACE in .env first"

def _config_set(key, value):
    subprocess.run(
        OPENCLAW_COMPOSE.split() + ["exec", "openclaw-gateway", "node", "/app/openclaw.mjs",
        "config", "set", key, value],
        capture_output=True, text=True
    )

_config_set("plugins.entries.opik-openclaw.config.enabled", "true")
_config_set("plugins.entries.opik-openclaw.config.apiKey", api_key)
_config_set("plugins.entries.opik-openclaw.config.workspaceName", workspace)
_config_set("plugins.entries.opik-openclaw.config.projectName", "compliance-agent")

subprocess.run(OPENCLAW_COMPOSE.split() + ["restart", "openclaw-gateway"],
               capture_output=True, text=True)

print("✅ Opik configured and gateway restarted.")

In [ ]:
import time
time.sleep(3)  # let gateway finish starting

result = subprocess.run(
    OPENCLAW_COMPOSE.split() + ["exec", "openclaw-gateway", "node", "/app/openclaw.mjs", "opik", "status"],
    capture_output=True, text=True
)
# Print only the Opik status block, strip noise
lines = result.stdout.splitlines()
in_status = False
for line in lines:
    if line.strip().startswith("Opik status:"):
        in_status = True
    if in_status and not "[plugins]" in line:
        print(line)

---
### What You're Looking At

Each message your agent handles becomes a **trace** in Opik. Click into one and you'll see:

| Field | What it tells you |
|-------|-------------------|
| **Input / Output** | Exactly what the user sent and what the agent replied |
| **Spans** | One span per LLM call — most turns are a single span, tool-using turns have more |
| **# tokens** | Input + output token counts for that span |
| **$ cost** | USD cost of the call — cumulative across spans gives you per-turn cost |
| **Duration** | Wall-clock time for each LLM call |
| **channel** | Where the message came from (`telegram`, `cli`, `dashboard`) |
| **trigger** | What initiated the turn — `user` for messages, `cron` for scheduled tasks |
| **sessionId** | Links all turns in a single conversation thread |
| **runId** | Unique ID for this specific agent invocation |

The **tag** `openclaw` is added automatically to every trace — use it to filter across projects.

> **Spans vs. traces:** A trace is one agent turn (one user message → one reply). Each LLM call *within* that turn is a span. A simple question = 1 span. A compliance check that calls Brave Search = 2+ spans: one for the search tool decision, one (or more) for the actual LLM reasoning with results.

### Why This Matters in Production

An autonomous agent running on a schedule is a **black box by default**. Opik gives you the window in:

**Cost control** — The token count on that "Hi can you hear me?" trace was ~13,000 input tokens. That's the system prompt + all of `AGENTS.md` loaded on every turn. Opik makes this visible so you can tune prompt size and cache aggressively before costs compound at scale.

**Debugging tool calls** — When a compliance check comes back wrong, you want to know: did the agent call the right skill? Did Brave Search return useful results? Did the LLM misread them? Opik's span tree shows each step with its inputs and outputs, so you can pinpoint exactly where the reasoning broke down.

**Scheduled task auditing** — When your agent runs at 3am on a cron schedule (trigger: `cron`), the trace is your receipt. You can verify it ran, what it found, and what it sent — without waking up.

**Feedback loop** — Opik's "Feedback scores" tab lets you annotate traces as good/bad. Over time this becomes your evaluation dataset for testing prompt changes before deploying them.